## Загрузка данных

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Ипорт библиотек

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from torch.utils.data import Dataset

In [ ]:
# --- Системные и общие ---
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import warnings
from collections import Counter
warnings.filterwarnings('ignore')

# --- SKlearn ---
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

# --- PyTorch & Transformers ---
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader


# --- Константы и настройка среды ---
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# Определяем устройство: MPS (Apple Silicon GPU), CUDA или CPU
if torch.backends.mps.is_available():
    device = torch.device("mps")
    torch.mps.manual_seed(RANDOM_STATE)
elif torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(RANDOM_STATE)
else:
    device = torch.device("cpu")

print(f"✅ Используемое устройство: {device}")

✅ Используемое устройство: cuda


In [ ]:
import torch
import torch.nn as nn

In [ ]:
train = pd.read_parquet('/content/drive/MyDrive/images/train.parquet')
test = pd.read_parquet('/content/drive/MyDrive/images/test.parquet')

In [ ]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

Класс Dataset для данных.

In [ ]:
from torch.utils.data import Dataset
from PIL import Image
import io


In [ ]:
class ImgesDataset(Dataset):
    def __init__(self, df, transform=None, train_flag=True):

      self.df = df.reset_index(drop=True)
      self.transform = transform
      self.train_flag = train_flag

    def __len__(self):
      return len(self.df)

    def __getitem__(self, idx):
      if self.train_flag:
        row = self.df.iloc[idx]
        img1_bytes = row['image_1']
        img2_bytes = row['image_2']
        label = row['is_image1_better']

        img1 = Image.open(io.BytesIO(img1_bytes)).convert('RGB')
        img2 = Image.open(io.BytesIO(img2_bytes)).convert('RGB')

        if self.transform:
          img1 = self.transform(img1)
          img2 = self.transform(img2)
        return img1, img2, label
      else:
        row = self.df.iloc[idx]
        img1_bytes = row['image_1']
        img2_bytes = row['image_2']

        img1 = Image.open(io.BytesIO(img1_bytes)).convert('RGB')
        img2 = Image.open(io.BytesIO(img2_bytes)).convert('RGB')

        if self.transform:
          img1 = self.transform(img1)
          img2 = self.transform(img2)
        else:
          img1 = transforms.ToTensor()(img1)
          img2 = transforms.ToTensor()(img2)
        return img1, img2



## Архитектура модели

Организуем 2 сверточных слоя и линейной головой. Далее используем сигмоиду на выходе, чтобы получить вероятность принадлежности к классу.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_c)
        )
        self.skip = nn.Sequential()
        if stride != 1 or in_c != out_c:
            self.skip = nn.Sequential(
                nn.Conv2d(in_c, out_c, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_c)
            )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        out = self.main(x)
        out += self.skip(x)
        return self.relu(out)

In [ ]:
class FeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )
        self.block1 = ConvBlock(32, 64, stride=2)
        self.block2 = ConvBlock(64, 128, stride=2)
        self.block3 = ConvBlock(128, 256, stride=2)
        self.pool   = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        x = self.stem(x)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.pool(x).flatten(1)
        return x

In [ ]:
class ComparisonHead(nn.Module):
    def __init__(self, feat_dim=256):
        super().__init__()
        in_dim = feat_dim * 4
        self.fc = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )

    def forward(self, f1, f2):
        x = torch.cat([f1, f2, torch.abs(f1 - f2), f1 * f2], dim=1)
        return self.fc(x)

In [ ]:
class PairwiseModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = FeatureExtractor()
        self.head = ComparisonHead(feat_dim=256)

    def forward(self, img1, img2):
        f1 = self.backbone(img1)
        f2 = self.backbone(img2)
        logits = self.head(f1, f2).squeeze(1)
        return logits

In [ ]:
model = PairwiseModel()

In [ ]:
x_train, x_val = train_test_split(train, test_size=0.2, random_state=RANDOM_STATE)

In [ ]:
train_dataset = ImgesDataset(x_train, transform=transform)
val_dataset = ImgesDataset(x_val, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
model = model.to(device)

NUM_EPOCHS =10

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    for img1, img2, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
        img1, img2, labels = img1.to(device), img2.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(img1, img2).squeeze()     # (B,)
        loss = criterion(logits, labels.float())
        probs = torch.sigmoid(logits)
        preds = (probs>0.5).float()
        acc = (preds==labels).float().mean().item()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Train Loss: {total_loss / len(train_loader):.4f}, acc:{acc}")

Epoch 1/10:   0%|          | 0/218 [00:00<?, ?it/s]

Train Loss: 0.6504, acc:0.625


Epoch 2/10:   0%|          | 0/218 [00:00<?, ?it/s]

Train Loss: 0.6438, acc:0.5833333730697632


Epoch 3/10:   0%|          | 0/218 [00:00<?, ?it/s]

...

Train Loss: 0.6384, acc:0.75


Epoch 9/10:   0%|          | 0/218 [00:00<?, ?it/s]

Train Loss: 0.6387, acc:0.4583333432674408


Epoch 10/10:   0%|          | 0/218 [00:00<?, ?it/s]

Train Loss: 0.6365, acc:0.625


In [ ]:
model.eval()

PairwiseModel(
  (backbone): FeatureExtractor(
    (stem): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
    )
    (block1): ConvBlock(
      (main): Sequential(
        (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (skip): Sequential(
        (0): Conv2d(32, 64, kernel_size=(1, 1), stride=(2, 2), bias=False)
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (relu): ReLU(inplace=True)
    )
    (block2): 

## Подбор лучшего трешхолда

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, roc_curve

# Предсказания
pred_log = []
y_true   = []

model.eval()
with torch.no_grad():
    for batch in tqdm(val_loader, desc="Predict"):
        if len(batch) == 3:
            img1, img2, labels = batch
            y_true.extend(labels.numpy())
        else:
            img1, img2 = batch
        img1, img2 = img1.to(device), img2.to(device)

        logits = model(img1, img2)
        probs  = torch.sigmoid(logits).view(-1).cpu().numpy()
        pred_log.extend(probs)

pred_log = np.array(pred_log)
y_true   = np.array(y_true, dtype=np.int64)

# AUC
auc = roc_auc_score(y_true, pred_log)
print(f"ROC-AUC: {auc:.4f}")

# Перебор трешхолдов
pred_stats = []
for i in range(1, 10):
    thr = i / 10.0
    y_pred = (pred_log >= thr).astype(int)
    f1  = f1_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    pred_stats.append((thr, f1, acc))

print("thr   F1      Acc")
for thr, f1, acc in pred_stats:
    print(f"{thr:.1f}  {f1:.4f}  {acc:.4f}")

# Лучший треш по F1
best_thr_f1, best_f1, _ = max(pred_stats, key=lambda x: x[1])
print(f"\nBest by F1: thr={best_thr_f1:.3f}, F1={best_f1:.4f}")

# Лучший по Youden'у (TPR - FPR)
fpr, tpr, thr_list = roc_curve(y_true, pred_log)
youden = tpr - fpr
idx = np.argmax(youden)
print(f"Best by Youden: thr={thr_list[idx]:.3f}, J={youden[idx]:.4f}, TPR={tpr[idx]:.4f}, FPR={fpr[idx]:.4f}")



Predict:   0%|          | 0/55 [00:00<?, ?it/s]

ROC-AUC: 0.5560
thr   F1      Acc
0.1  0.5256  0.3565
0.2  0.5256  0.3565
0.3  0.5250  0.3904
0.4  0.4317  0.5465
0.5  0.0685  0.6406
0.6  0.0000  0.6435
0.7  0.0000  0.6435
0.8  0.0000  0.6435
0.9  0.0000  0.6435

Best by F1: thr=0.100, F1=0.5256
Best by Youden: thr=0.378, J=0.0959, TPR=0.6490, FPR=0.5531


## Формирование сабмита

In [ ]:
test_dataset = ImgesDataset(test, transform=transform, train_flag=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
pred_log = []

with torch.no_grad():
    for img1, img2 in tqdm(test_loader, desc="Predict"):
        img1, img2 = img1.to(device), img2.to(device)
        logits = model(img1, img2)
        probs  = torch.sigmoid(logits).view(-1).cpu().numpy()
        pred_log.extend(probs)

Predict:   0%|          | 0/135 [00:00<?, ?it/s]

In [ ]:
pred_log=np.array(pred_log)

In [ ]:
(pred_log > 0.378).astype(int)

array([1, 1, 1, ..., 1, 0, 1])

In [ ]:
def sub(pred):
    sap = pd.read_csv('sample_submission.csv', index_col='index')
    sap['is_image1_better']=pred
    sap.to_csv('sub.csv')

In [ ]:
sub((pred_log > 0.395).astype(int))